# Scheme C: Classification → probability distribution over K


In [ ]:
# Assumes: pip install -e . from project root

In [ ]:
### Imports
import numpy as np
import torch
import matplotlib.pyplot as plt
from spectackle.config import deep_update, set_cpu_safety
from spectackle.data import BASE_CFG, make_loaders
from spectackle.models import CountNet1D_Classify
from spectackle.training import train_scheme_c
from spectackle.plotting import plot_example, collect_count_predictions
set_cpu_safety(1)  ### Avoid CPU oversubscription on macOS

In [ ]:
### Config + data
### k_mode="uniform" → equal probability across 0..Kmax (harder than poisson default)
cfg = deep_update(BASE_CFG, dict(
    gen=dict(
        k_mode="uniform",
        p_zero=0.0,
        k_tail_prob=0.0,
    )
))
Kmax = int(cfg["max_components"])
train_loader, val_loader = make_loaders(cfg)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kmax={Kmax}, device={device}")

In [ ]:
### Plot one example spectrum
plot_example(val_loader.dataset, idx=0)

In [ ]:
### Model: baseline Scheme C CNN
model = CountNet1D_Classify(Kmax=Kmax, width=64)

In [ ]:
### Train
model = train_scheme_c(
    model, train_loader, val_loader,
    device=device, lr=1e-3, epochs=3, log_every=200,
)

In [ ]:
### Diagnostics

y_true, y_pred, y_exp = collect_count_predictions(
    model, val_loader, device=device, Kmax=Kmax
)
y_show = y_pred  ### swap to y_exp for E[K] view

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
edges = np.arange(-0.5, Kmax + 1.5, 1.0)
ticks = np.arange(0, Kmax + 1)

### raw counts
H, _, _ = np.histogram2d(y_true, y_show, bins=[edges, edges])
im = axes[0].imshow(H.T, origin="lower", aspect="equal",
                    interpolation="nearest",
                    extent=[edges[0], edges[-1], edges[0], edges[-1]])
axes[0].set_xlabel("K true"); axes[0].set_ylabel("K pred")
axes[0].set_title("True vs Pred (count)")
axes[0].set_xticks(ticks); axes[0].set_yticks(ticks)
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="count")

### row-normalised: P(K_pred | K_true)
Hn = H / (H.sum(axis=1, keepdims=True) + 1e-12)
im2 = axes[1].imshow(Hn.T, origin="lower", aspect="equal",
                     interpolation="nearest",
                     extent=[edges[0], edges[-1], edges[0], edges[-1]])
axes[1].set_xlabel("K true"); axes[1].set_ylabel("K pred")
axes[1].set_title("True vs Pred (row-normalised)")
axes[1].set_xticks(ticks); axes[1].set_yticks(ticks)
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04,
             label="P(K_pred | K_true)")

### error histogram
err = y_show - y_true
bins_e = np.arange(err.min() - 0.5, err.max() + 1.5, 1.0)
axes[2].hist(err, bins=bins_e, color="cornflowerblue", alpha=0.5)
axes[2].hist(err, bins=bins_e, histtype="step", color="k", lw=2)
axes[2].set_xlabel("K_pred - K_true"); axes[2].set_title("Count error histogram")

plt.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
### MAE vs K_true and K_true frequency

ks = np.arange(0, Kmax + 1)
mae_by_k = [
    np.mean(np.abs(y_show[y_true == k] - k)) if np.any(y_true == k) else np.nan
    for k in ks
]

plt.figure(figsize=(6, 3))
plt.plot(ks, mae_by_k, marker="o")
plt.xlabel("K_true"); plt.ylabel("MAE"); plt.title("MAE vs K_true")
plt.tight_layout(); plt.show()

counts = np.array([(y_true == k).sum() for k in ks])
plt.figure(figsize=(6, 3))
plt.bar(ks, counts)
plt.xlabel("K_true"); plt.ylabel("count")
plt.title("K_true frequency in eval sample")
plt.tight_layout(); plt.show()